In [ ]:
import numpy as np
from aeon.classification.interval_based import RSTSF
from sklearn.metrics import accuracy_score
from tscglue.models import LokyStackerV7, LokyStackerV7SoftFilterRidge, TSCGlue, LokyStackerV8Base, LokyStackerV8AutoBestStacking, LokyStackerV8AutoBestBase
from tscglue import utils, data_loader
import polars as pl

In [ ]:
dataset = "Worms"
# dataset = 'Car'
dataset = 'HandOutlines'
dataset = 'Trace'
dataset = 'SwedishLeaf'
X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, 0)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

In [ ]:
m = LokyStackerV8Base(random_state=492, n_repetitions=1, n_jobs=16, keep_features=True, verbose=2)
m.fit(X_train, y_train)

In [ ]:
preds = m.predict_per_model(X_test)
tests_accs = []
for model_name, model_preds in preds.items():
    acc = accuracy_score(y_test, model_preds)
    tests_accs.append({"model": model_name, "test_accuracy": acc})
test_df = pl.DataFrame(tests_accs)

In [ ]:
pl.DataFrame(m.summary())

In [ ]:
pl.DataFrame(m.summary()).join(test_df, on="model").sort("test_accuracy", descending=True)

In [ ]:
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc}")

In [ ]:
F = m.get_features()
F.estimated_size('gb')

In [ ]:
P = m.get_oof_predictions()
P.estimated_size('gb')